<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.1: Procesamiento de Textos y Bag of Words

> **Prof. Heider Sanchez**  

##  Introducción
Este laboratorio tiene como objetivo el análisis y búsqueda de documentos textuales utilizando procesamiento de lenguaje natural (NLP) y una base de datos PostgreSQL. Se trabajará paso a paso desde la extracción de los textos hasta la aplicación búsquedas booleanas.


### Objetivos
- Configurar la tabla en PostgreSQL y carga de datos.
- Desde Python leer los textos desde PostgreSQL.
- Realizar el procesamiento de textos: convertir a minúscula, tokenización, stopwords, stemming y frecuencia de términos.
- Almacenar los Bag of Words en la base de datos en formato JSON.
- Realizar búsquedas de documentos similares a una consulta booleana (conectores AND, OR y AND-NOT).


### Requisitos previos

- Tener instalado PostgreSQL en su computadora (ultima versión)
- Tener instalado las siguientes dependencias en Python:

    ```bash
    pip install psycopg2-binary nltk scikit-learn pandas
    ```

- Opcionalmente descargar los recursos de NLTK:

    ```python
    import nltk
    nltk.download('punkt')
    ```


## 1. (2 puntos) Configurar la tabla en PostgreSQL y carga de datos


### Crear las tablas

Crear la tabla en PostgreSQL para almacenar los textos de noticias y el bag of words:

```sql
CREATE TABLE noticias (
    id SERIAL PRIMARY KEY,
    url TEXT,
    contenido TEXT,
    categoria VARCHAR(50),
    bag_of_words JSONB
);
```

Además, crear una tabla para almacenar los stopwords

```sql
CREATE TABLE stopwords (
    id SERIAL PRIMARY KEY,
    word TEXT UNIQUE NOT NULL
);
```

### Carga de datos en PostgreSQL

Proceder a cargar el dataset de noticias `news_es.csv` y el dataset de stopwords `stoplist_es.txt`.

### Leer desde PostgreSQL con Python

Completar la función para conectarte a PostgreSQL y leer los datos:

In [3]:
%pip install psycopg2-binary pandas nltk scikit-learn

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ------- -------------------------------


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Exception:
Traceback (most recent call last):
  File "c:\Users\Rodrigo Zambrano\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "c:\Users\Rodrigo Zambrano\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rodrigo Zambrano\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rodrigo Zambrano\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.r

In [4]:
import psycopg2
import pandas as pd

def connect_db():
    conn = psycopg2.connect(
        dbname="lab8",
        user="postgres",
        password="admin",
        host="localhost",
        port="5433"
    )
    return conn

def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido FROM noticias;"
    df = pd.read_sql(query, conn)
    conn.close()
    return df

noticias_df = fetch_data()


C:\Users\Rodrigo Zambrano\AppData\Local\Temp\ipykernel_3132\1443427590.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## 2. (4 puntos) Preprocesamiento de texto

Implementar la función `preprocess` que reciba un texto y realice los siguiente:
- Convertir el texto a minuscula.
- Tokenización.
- Eliminación de stopwords
- Stemming (raíz de las palabras)

In [5]:
import nltk
from nltk.stem import SnowballStemmer

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to C:\Users\Rodrigo
[nltk_data]     Zambrano\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Rodrigo
[nltk_data]     Zambrano\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [6]:

def load_stopwords():
    conn = connect_db()
    cursor = conn.cursor()
    cursor.execute("SELECT word FROM stopwords;")
    stopwords_set = set(row[0] for row in cursor.fetchall())
    cursor.close()
    conn.close()
    return stopwords_set

stopwords_es = load_stopwords()
stemmer = SnowballStemmer('spanish')

def preprocess(text):
    text = text.lower()

    text = text.replace('¡', '').replace('¿', '')

    raw_tokens = nltk.word_tokenize(text)
    
    filtered_words = [
        token for token in raw_tokens 
        if token.isalnum() and token not in stopwords_es
    ]

    stemmed_words = [stemmer.stem(word) for word in filtered_words]
    
    return stemmed_words

print(preprocess("¡Hola! Esto es una prueba, con números 123 y símbolos #@$%. ¿Funcionará bien?"))


['hol', 'prueb', 'numer', '123', 'simbol', 'funcion']


Luego, implementar una función para calcular la frecuencia de términos:

In [7]:
def compute_bow(text):
    tokens = preprocess(text)
    
    bow = {}
    for token in tokens:
        if token in bow:
            bow[token] += 1
        else:
            bow[token] = 1
            
    return bow

print(compute_bow("Esta es una prueba con prueba y más prueba. ¡Prueba!. Programadores programadores programando."))

{'prueb': 4, 'program': 3}


## 3. (3 puntos) Actualizar la base de datos con los Bag of Words

Guardar el resultado del Bag of Words en la columna `bag_of_words` de la tabla:

In [8]:
import json

def update_bow_in_db(dataframe):
    conn = connect_db()
    cursor = conn.cursor()
    
    try:
        print("Calculando BoW y actualizando la base de datos...")
        
        for index, row in dataframe.iterrows():
            id_noticia = row['id']
            contenido_noticia = row['contenido']
            
            bow_dict = compute_bow(contenido_noticia)
            
            bow_json = json.dumps(bow_dict)
            
            query = "UPDATE noticias SET bag_of_words = %s WHERE id = %s;"
            cursor.execute(query, (bow_json, id_noticia))
            
        conn.commit()
        print("Todos los registros de bag_of_words se actualizaron correctamente!")
        
    except Exception as e:
        conn.rollback()
        print(f"Ocurrió un error al actualizar: {e}")
        
    finally:
        cursor.close()
        conn.close()

update_bow_in_db(noticias_df)

Calculando BoW y actualizando la base de datos...
Todos los registros de bag_of_words se actualizaron correctamente!


In [9]:
def fetch_data_bow():
    conn = connect_db()
    
    query = "SELECT id, bag_of_words FROM noticias LIMIT 10;"
    
    df_bow = pd.read_sql(query, conn)
    
    conn.close()
    
    return df_bow

fetch_data_bow()

C:\Users\Rodrigo Zambrano\AppData\Local\Temp\ipykernel_3132\4101556451.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bow = pd.read_sql(query, conn)


,id,bag_of_words
0,98,"{'3': 1, '4': 1, '5': 1, '6': 1, '8': 1, '01':..."
1,65,"{'11': 1, '73': 1, 'cb': 1, 'ia': 3, 'of': 1, ..."
2,79,"{'z': 1, '50': 1, '56': 1, 'b2b': 1, 'cre': 1,..."
3,86,"{'1': 2, '2': 1, '3': 2, '4': 1, '40': 1, '50'..."
4,91,"{'av': 2, '035': 1, '100': 1, '146': 1, '171':..."
5,80,"{'1': 2, '5': 1, '100': 1, '150': 1, 'hor': 1,..."
6,685,"{'1': 1, '10': 1, '25': 1, '34': 1, 'ceo': 1, ..."
7,87,"{'cas': 1, 'cdt': 2, 'fij': 1, 'lig': 1, 'pag'..."
8,142,"{'50': 1, '442': 1, 'agu': 5, 'aun': 1, 'baj':..."
9,145,{}


## 4. (8 puntos) Consulta booleana con filtrado por keywords

Antes de aplicar el filtrado desde Python, es importante entender cómo funciona la consulta de una clave dentro de una columna JSONB en PostgreSQL. 

### Ejemplo consulta SQL en JSON:

```sql
SELECT * FROM noticias WHERE bag_of_words ? 'keyword';
```

Esta consulta selecciona todos los registros en los que el `bag_of_words` (formato JSONB) contiene una clave igual a `'keyword'`. El operador `?` verifica la existencia de una clave dentro de un JSON.

### Consulta booleana 
Implementar una función que permita parsear una consulta textual con conectores AND, OR y AND-NOT y con ello se aplique el filtro correspondiente directamente desde la base de datos. 


In [10]:
def apply_boolean_query(query_text):
    conn = connect_db()

    parts = query_text.split()
    
    sql_conditions = []
    
    i = 0

    while i < len(parts):
        token = parts[i]
        
        if token == "AND":
            sql_conditions.append("AND")
        elif token == "OR":
            sql_conditions.append("OR")
        elif token == "AND-NOT":
            sql_conditions.append("AND NOT")
        else:

            stems = preprocess(token)
            if stems:
                term = stems[0]
                sql_conditions.append(f"bag_of_words ? '{term}'")
            else:
                pass
        
        i += 1


    if not sql_conditions:
        print("Consulta vacía o solo contiene stopwords.")
        return pd.DataFrame()

    where_clause = " ".join(sql_conditions)
    final_query = f"SELECT id, contenido FROM noticias WHERE {where_clause};"
    
    print(f"DEBUG SQL: {final_query}") 
    
    try:
        df_results = pd.read_sql(final_query, conn)
        return df_results
    except Exception as e:
        print(f"Error en la ejecución SQL: {e}")
        return pd.DataFrame()
    finally:
        conn.close()

### Pruebas funcionales

Realizar al menos 8 pruebas funcionales con mas de dos keywords de consulta:

In [11]:
test_queries = [
    "transformación AND sostenible",
    "México OR Perú",
    "economía AND-NOT crisis",
    "tecnología AND futuro",
    "deporte OR cultura",
    "gobierno AND inversión",
    "salud AND-NOT enfermedad",
    "educación AND digital"
]

for q in test_queries:
    print(f"Buscando: {q}")
    res = apply_boolean_query(q)
    
    if not res.empty:
        total = len(res)
        print(f"Encontrados {total} documentos.")
        print("-" * 15)
        
        for index, row in res.head(3).iterrows():
            print(f"ID: {row['id']}")
            print(f"Texto: {row['contenido'][:200]}...") 
            print("." * 10)
            
    else:
        print("Sin resultados.")
    
    print("-" * 30)

Buscando: transformación AND sostenible
DEBUG SQL: SELECT id, contenido FROM noticias WHERE bag_of_words ? 'transform' AND bag_of_words ? 'sosten';
Encontrados 68 documentos.
---------------
ID: 278
Texto: El suprarreciclaje es transformar un desecho en un producto de mayor calidad y valor ecológico. Conocido como ‘upcycling’, esta estrategia integrada en la economía circular es un paso más en el proces...
..........
ID: 677
Texto: De las fuentes de energía renovable oceánicas, la generada por la corriente marina puede considerarse la más madura y cercana a la comercialización. Allí donde se dé una diferencia mínima de cinco met...
..........
ID: 714
Texto: El nuevo Marco de Competencia Digital Docente ha acelerado el reloj en los centros educativos de España. De aquí a 2024, el país deberá certificar las competencias digitales de, al menos, el 80 % de l...
..........
------------------------------
Buscando: México OR Perú
DEBUG SQL: SELECT id, contenido FROM noticias WHERE bag_of_word

C:\Users\Rodrigo Zambrano\AppData\Local\Temp\ipykernel_3132\1981608983.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_results = pd.read_sql(final_query, conn)


DEBUG SQL: SELECT id, contenido FROM noticias WHERE bag_of_words ? 'deport' OR bag_of_words ? 'cultur';
Encontrados 81 documentos.
---------------
ID: 246
Texto: La evolución tecnológica, los procesos de digitalización de las distintas industrias y el nuevo contexto sociolaboral de teletrabajo hacen de la ciberdelincuencia un fenómeno cada vez más extendido y ...
..........
ID: 369
Texto: La gamificación (o ludificación) es una técnica de aprendizaje que fomenta la adquisición o consolidación de conocimientos mediante juegos, estimulando la motivación y la productividad. España es una ...
..........
ID: 403
Texto: La diversidad biológica es la vida. Desde las estructuras invisibles como los genes a las superestructuras de los ecosistemas, pasando por todas y cada una de las especies animales, vegetales, de hong...
..........
------------------------------
Buscando: gobierno AND inversión
DEBUG SQL: SELECT id, contenido FROM noticias WHERE bag_of_words ? 'gobiern' AND bag_of_words ? 'in

## 5. (3 puntos) Actividad Final
- Medir el tiempo de ejecución de las consultas con diferentes tamaños de datos y optimizar el código según sea necesario.


**Entregable:** informe de los resultados obtenidos en formato PDF.

Para poder evaluar el rendimiento en diferentes tamaños de datos se modificó la función original *apply_boolean_query* agregando un nuevo parámetro *limite_datos*, que permite filtrar por el *id*, escaneando solo aquellos menores al *limite_datos* determinado.

In [18]:
import time
import math

def apply_boolean_query_sized(query_text, limite_datos):
    conn = connect_db()
    parts = query_text.split()
    sql_conditions = []
    i = 0

    while i < len(parts):
        token = parts[i]
        if token == "AND":
            sql_conditions.append("AND")
        elif token == "OR":
            sql_conditions.append("OR")
        elif token == "AND-NOT":
            sql_conditions.append("AND NOT")
        else:
            stems = preprocess(token)
            if stems:
                term = stems[0]
                sql_conditions.append(f"bag_of_words ? '{term}'")
        i += 1

    if not sql_conditions:
        return pd.DataFrame()

    where_clause = " ".join(sql_conditions)
    
    #límite para simular el tamaño de la BD
    final_query = f"SELECT id, contenido FROM noticias WHERE ({where_clause}) AND id <= {limite_datos};"
    
    try:
        df_results = pd.read_sql(final_query, conn)
        return df_results
    except Exception as e:
        print(f"Error en la ejecución SQL: {e}")
        return pd.DataFrame()
    finally:
        conn.close()

Primero se consulta el total de registros en la tabla *noticias* para luego calcular cuatro umbrales de tamaño de datos (25%, 50%, 75% y 100% del conjunto de datos). Se ejecutan las consultas para cada tamaño y se registra el tiempo de ejecución en un nuevo DataFrame *resultados*.

In [19]:
import math
import time
consultas_prueba = [
    "transformación AND sostenible",
    "México OR Perú",
    "economía AND-NOT crisis",
    "tecnología AND futuro",
    "deporte OR cultura",
    "gobierno AND inversión",
    "salud AND-NOT enfermedad",
    "educación AND digital"
]

def obtener_total_filas():
    conn = connect_db()
    try:
        df_count = pd.read_sql("SELECT COUNT(*) as total FROM noticias;", conn)
        return int(df_count['total'][0])
    except Exception as e:
        print(f"Error al contar: {e}")
        return 0
    finally:
        conn.close()

total_real = obtener_total_filas()
tamanos_datos = [
    math.ceil(total_real * 0.25),
    math.ceil(total_real * 0.50),
    math.ceil(total_real * 0.75),
    total_real
]

def medir_rendimiento(etiqueta):
    print(f"{etiqueta}")
    resultados = []
    for size in tamanos_datos:
        for q in consultas_prueba:
            inicio = time.time()
            df_res = apply_boolean_query_sized(q, size) 
            fin = time.time()
            
            resultados.append({
                "Tamaño Datos": size,
                "Consulta": q,
                "Tiempo (s)": round(fin - inicio, 4),
                "Encontrados": len(df_res)
            })
    return pd.DataFrame(resultados)

C:\Users\Rodrigo Zambrano\AppData\Local\Temp\ipykernel_3132\2183436388.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql("SELECT COUNT(*) as total FROM noticias;", conn)


In [20]:
df= medir_rendimiento("MEDICIÓN SIN ÍNDICE (Sequential Scan)")
display(df)

MEDICIÓN SIN ÍNDICE (Sequential Scan)


C:\Users\Rodrigo Zambrano\AppData\Local\Temp\ipykernel_3132\2605304602.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_results = pd.read_sql(final_query, conn)


,Tamaño Datos,Consulta,Tiempo (s),Encontrados
0,305,transformación AND sostenible,0.0467,20
1,305,México OR Perú,0.0364,62
2,305,economía AND-NOT crisis,0.0506,126
3,305,tecnología AND futuro,0.0303,33
4,305,deporte OR cultura,0.0315,13
5,305,gobierno AND inversión,0.0306,16
6,305,salud AND-NOT enfermedad,0.0290,36
7,305,educación AND digital,0.0244,11
8,609,transformación AND sostenible,0.0586,34
9,609,México OR Perú,0.0312,143


| | Tamaño Datos | Consulta | Tiempo (s) | Encontrados |
|---|---|---|---|---|
| **0** | 305 | transformación AND sostenible | 0.0467 | 20 |
| **1** | 305 | México OR Perú | 0.0364 | 62 |
| **2** | 305 | economía AND-NOT crisis | 0.0506 | 126 |
| **3** | 305 | tecnología AND futuro | 0.0303 | 33 |
| **4** | 305 | deporte OR cultura | 0.0315 | 13 |
| **5** | 305 | gobierno AND inversión | 0.0306 | 16 |
| **6** | 305 | salud AND-NOT enfermedad | 0.0290 | 36 |
| **7** | 305 | educación AND digital | 0.0244 | 11 |
| **8** | 609 | transformación AND sostenible | 0.0586 | 34 |
| **9** | 609 | México OR Perú | 0.0312 | 143 |
| **10** | 609 | economía AND-NOT crisis | 0.0426 | 247 |
| **11** | 609 | tecnología AND futuro | 0.0404 | 60 |
| **12** | 609 | deporte OR cultura | 0.0275 | 34 |
| **13** | 609 | gobierno AND inversión | 0.0247 | 32 |
| **14** | 609 | salud AND-NOT enfermedad | 0.0257 | 69 |
| **15** | 609 | educación AND digital | 0.0235 | 19 |
| **16** | 913 | transformación AND sostenible | 0.0273 | 44 |
| **17** | 913 | México OR Perú | 0.0318 | 197 |
| **18** | 913 | economía AND-NOT crisis | 0.0548 | 378 |
| **19** | 913 | tecnología AND futuro | 0.0321 | 79 |
| **20** | 913 | deporte OR cultura | 0.0342 | 59 |
| **21** | 913 | gobierno AND inversión | 0.0244 | 50 |
| **22** | 913 | salud AND-NOT enfermedad | 0.0437 | 111 |
| **23** | 913 | educación AND digital | 0.0444 | 24 |
| **24** | 1217 | transformación AND sostenible | 0.0382 | 68 |
| **25** | 1217 | México OR Perú | 0.0448 | 257 |
| **26** | 1217 | economía AND-NOT crisis | 0.0527 | 512 |
| **27** | 1217 | tecnología AND futuro | 0.0371 | 108 |
| **28** | 1217 | deporte OR cultura | 0.0328 | 81 |
| **29** | 1217 | gobierno AND inversión | 0.0464 | 70 |
| **30** | 1217 | salud AND-NOT enfermedad | 0.0279 | 150 |
| **31** | 1217 | educación AND digital | 0.0223 | 31 |

De la primera medición se observa un cuello de botella, pues al no existir un índice, se realiza un Sequential Scan que ralentiza el proceso a medida que crece el tamaño de datos. Para solucionar esto, se crea un índice GIN sobre la columna *bag_of_words*. Así, en vez de escanear toda la tabla, el índice crea un mapa lógico que asocia cada token con los *id* de los documentos que lo contienen, disminuyendo considerablemente el tiempo de ejecución, como se puede observar en la siguiente tabla.

In [21]:
def optimizar_base_datos():
    conn = connect_db()
    conn.autocommit = True
    cursor = conn.cursor()
    try:
        print("Creando índice GIN")
        # Creamos el índice para acelerar el operador '?' en JSONB
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_noticias_bag_of_words ON noticias USING GIN (bag_of_words);")
        print("Índice GIN creado con éxito\n")
    except Exception as e:
        print(f"Error al crear índice: {e}")
    finally:
        cursor.close()
        conn.close()

optimizar_base_datos()

Creando índice GIN
Índice GIN creado con éxito



In [23]:
df_optimizado = medir_rendimiento("MEDICIÓN CON ÍNDICE GIN (Index Scan)")
display(df_optimizado)

MEDICIÓN CON ÍNDICE GIN (Index Scan)


C:\Users\Rodrigo Zambrano\AppData\Local\Temp\ipykernel_3132\2605304602.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_results = pd.read_sql(final_query, conn)


,Tamaño Datos,Consulta,Tiempo (s),Encontrados
0,305,transformación AND sostenible,0.0228,20
1,305,México OR Perú,0.0211,62
2,305,economía AND-NOT crisis,0.0234,126
3,305,tecnología AND futuro,0.0315,33
4,305,deporte OR cultura,0.0209,13
5,305,gobierno AND inversión,0.0185,16
6,305,salud AND-NOT enfermedad,0.0226,36
7,305,educación AND digital,0.0177,11
8,609,transformación AND sostenible,0.0316,34
9,609,México OR Perú,0.0227,143


| | Tamaño Datos | Consulta | Tiempo (s) | Encontrados |
|---|---|---|---|---|
| **0** | 305 | transformación AND sostenible | 0.0228 | 20 |
| **1** | 305 | México OR Perú | 0.0211 | 62 |
| **2** | 305 | economía AND-NOT crisis | 0.0234 | 126 |
| **3** | 305 | tecnología AND futuro | 0.0315 | 33 |
| **4** | 305 | deporte OR cultura | 0.0209 | 13 |
| **5** | 305 | gobierno AND inversión | 0.0185 | 16 |
| **6** | 305 | salud AND-NOT enfermedad | 0.0226 | 36 |
| **7** | 305 | educación AND digital | 0.0177 | 11 |
| **8** | 609 | transformación AND sostenible | 0.0316 | 34 |
| **9** | 609 | México OR Perú | 0.0227 | 143 |
| **10** | 609 | economía AND-NOT crisis | 0.0308 | 247 |
| **11** | 609 | tecnología AND futuro | 0.0319 | 60 |
| **12** | 609 | deporte OR cultura | 0.0189 | 34 |
| **13** | 609 | gobierno AND inversión | 0.0194 | 32 |
| **14** | 609 | salud AND-NOT enfermedad | 0.0198 | 69 |
| **15** | 609 | educación AND digital | 0.0216 | 19 |
| **16** | 913 | transformación AND sostenible | 0.0199 | 44 |
| **17** | 913 | México OR Perú | 0.0264 | 197 |
| **18** | 913 | economía AND-NOT crisis | 0.0523 | 378 |
| **19** | 913 | tecnología AND futuro | 0.0390 | 79 |
| **20** | 913 | deporte OR cultura | 0.0215 | 59 |
| **21** | 913 | gobierno AND inversión | 0.0267 | 50 |
| **22** | 913 | salud AND-NOT enfermedad | 0.0418 | 111 |
| **23** | 913 | educación AND digital | 0.0373 | 24 |
| **24** | 1217 | transformación AND sostenible | 0.0229 | 68 |
| **25** | 1217 | México OR Perú | 0.0288 | 257 |
| **26** | 1217 | economía AND-NOT crisis | 0.0399 | 512 |
| **27** | 1217 | tecnología AND futuro | 0.0239 | 108 |
| **28** | 1217 | deporte OR cultura | 0.0287 | 81 |
| **29** | 1217 | gobierno AND inversión | 0.0192 | 70 |
| **30** | 1217 | salud AND-NOT enfermedad | 0.0290 | 150 |
| **31** | 1217 | educación AND digital | 0.0188 | 31 |

Al comparar los resultados de ambas tablas, se evidencia una mejora clara en los tiempos después la implementación del índice GIN, notándose más en el conjunto de datos de mayor tamaño (1,217 registros).